# Datagen trace viewer

Step through a **recorded** datagen run (`data_game/<label>/`) one move at a
time — the sanity-check companion to `interactive_self_eval.ipynb`. Same
rendering (frame, searches, player reply, verified wrong-span highlights,
analyst analysis, rating), but nothing is generated: everything comes from
`traces.jsonl` + `analyst_traces.jsonl` + the stable frame copies in
`images/`, exactly as training will see them.

Controls:

- **Run** / **Game** dropdowns at the top pick a datagen label and one of its
  games (the game list shows move count, win/loss, and the session id, since
  `--append` resumes can reuse game numbers under a new session).
- **Reset** starts stepping the selected game from move 0 (press it after
  changing the dropdowns).
- **Next move** (bottom, like the interactive notebook) prints the next
  round: the player question, the frame the player saw, the player reply,
  the analyst question, and the analyst analysis.

No GPU, no NAMS, no model — this notebook only reads files, so it runs
anywhere the repo and the `data_game/` directory exist. Handy for the manual
checks in `training/TO_TEST.md` (noised-frame readability, win stamping,
question-round analyses).

In [3]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

# Run from the repo root so data_game/ resolves (same as the other notebooks).
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

DATA_GAME = Path("data_game")


def available_runs() -> list[str]:
    """Datagen labels under data_game/ that have a traces.jsonl."""
    if not DATA_GAME.is_dir():
        return []
    return sorted(
        p.name for p in DATA_GAME.iterdir()
        if p.is_dir() and (p / "traces.jsonl").is_file()
    )


def _read_jsonl(path: Path) -> list[dict]:
    records = []
    if path.is_file():
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
    return records


def load_run(label: str):
    """One run's records, arranged for stepping.

    Returns ``(games, analyst)``: ``games`` maps a game key
    ``(session_id, game_index)`` to that game's player records in move
    order; ``analyst`` maps ``(session_id, game_index, move_index)`` to the
    matching analyst record. The session id is part of the key because
    ``--append`` resumes (e.g. run_weekend retries) restart game numbering
    under a fresh session, so game_index alone is not unique."""
    run_dir = DATA_GAME / label
    games: dict[tuple, list[dict]] = {}
    for rec in _read_jsonl(run_dir / "traces.jsonl"):
        meta = rec["meta"]
        games.setdefault((meta["session_id"], meta["game_index"]),
                         []).append(rec)
    for recs in games.values():
        recs.sort(key=lambda r: r["meta"]["move_index"])
    analyst = {
        (rec["meta"]["session_id"], rec["meta"]["game_index"],
         rec["meta"]["move_index"]): rec
        for rec in _read_jsonl(run_dir / "analyst_traces.jsonl")
    }
    return games, analyst


def frame_path(label: str, record: dict) -> Path | None:
    """This move's stable frame copy: datagen rewrites exactly the frame the
    player saw to an ``images/g####_m###_*`` url (earlier context frames keep
    their original session paths). Prefer the url stamped with this record's
    own game/move numbers; fall back to the last images/ url."""
    meta = record["meta"]
    want = f"g{meta['game_index']:04d}_m{meta['move_index']:03d}_"
    hits = []
    for m in record["messages"]:
        content = m.get("content")
        if not isinstance(content, list):
            continue
        for part in content:
            if (isinstance(part, dict) and part.get("type") == "image"
                    and str(part.get("url", "")).startswith("images/")):
                hits.append(str(part["url"]))
    for url in hits:
        if Path(url).name.startswith(want):
            return DATA_GAME / label / url
    return (DATA_GAME / label / hits[-1]) if hits else None


runs = available_runs()
print(f"{len(runs)} datagen run(s) found:", ", ".join(runs) or "(none)")

26 datagen run(s) found: aug5_iter1, aug5_iter2, aug5_smoke1, aug5_smoke2, aug6_iter1, aug6_iter10, aug6_iter11, aug6_iter2, aug6_iter3, aug6_iter4, aug6_iter5, aug6_iter6, aug6_iter7, aug6_iter8, aug6_iter9, aug6_smoke1, aug6_smoke10, aug6_smoke11, aug6_smoke2, aug6_smoke3, aug6_smoke4, aug6_smoke5, aug6_smoke6, aug6_smoke7, aug6_smoke8, aug6_smoke9


In [4]:
import html as _html

import ipywidgets as widgets
from IPython.display import HTML, Image, clear_output, display

# Frames shown at the same fixed width as interactive_self_eval.ipynb.
FRAME_WIDTH = 420  # px

# ------------------------------------------------------------- top controls
run_dd = widgets.Dropdown(
    options=available_runs(), description="Run:",
    layout=widgets.Layout(width="460px"),
)
game_dd = widgets.Dropdown(
    options=[], description="Game:",
    layout=widgets.Layout(width="700px"),
)
reset_btn = widgets.Button(description="Reset", button_style="warning")

# -------------------------------------------------------------- the stepper
next_btn = widgets.Button(description="Next move", button_style="primary")
out = widgets.Output()

# Current run's data + stepping position (module-level state, like the
# round_has_analysis flag in the interactive notebook).
_state = {"analyst": {}, "moves": [], "pos": 0}


def _game_option(key: tuple, recs: list[dict]) -> str:
    session_id, game_idx = key
    won = recs[-1]["meta"].get("game_won")
    return (f"game {game_idx} -- {len(recs)} move(s), "
            f"{'WON' if won else 'lost'} (session {session_id[:8]})")


def _refresh_games(*_):
    """(Re)load the selected run and repopulate the game dropdown."""
    if run_dd.value is None:
        game_dd.options = []
        return
    games, analyst = load_run(run_dd.value)
    _state["analyst"] = analyst
    _state["all_games"] = games
    game_dd.options = [
        (_game_option(k, v), k)
        for k, v in sorted(games.items(), key=lambda kv: (kv[0][1], kv[0][0]))
    ]


def _show_frame(path, caption):
    print(caption)
    if path is not None and Path(path).is_file():
        display(Image(filename=str(path), width=FRAME_WIDTH))
    else:
        print(f"  [frame missing on disk: {path}]")


def _show_wrong_spans(player_raw, spans):
    """Verified wrong spans highlighted in the player reply -- the same
    rendering as interactive_self_eval.ipynb (spans here are already
    harness-verified: datagen stores verified/unverified separately)."""
    if not spans:
        return
    marked = _html.escape(player_raw)
    for span in spans:
        marked = marked.replace(
            _html.escape(span), "<mark>" + _html.escape(span) + "</mark>",
        )
    print("\n-- player reply with VERIFIED wrong spans highlighted --")
    display(HTML(
        "<div style='white-space: pre-wrap; border-left: 3px solid #c00; "
        "padding-left: 8px;'>" + marked + "</div>"
    ))


def _show_move(record):
    """One full round, in conversation order: player question, frame,
    searches, player reply, analyst question, analysis, rating."""
    meta = record["meta"]
    k, n = meta["move_index"] + 1, len(_state["moves"])
    print(f"\n{'=' * 24} move {k}/{n} {'=' * 24}")

    # --- user stimulus
    print(f"=== You: {meta['question']}")
    _show_frame(frame_path(run_dd.value, record), "-- frame the player saw --")

    # --- player response
    for s in meta.get("searches", []):
        print(f"  [SEARCH {s['query']}]")
    print(f"Player: {record['target_text']}")
    if meta.get("action"):
        print(f"[move recorded: {meta['action']}"
              + ("  -- gold collected!" if meta.get("gold_collected") else "")
              + "]")
    elif meta.get("bare_move"):
        print(f"[FORMAT ERROR: bare '{meta['bare_move']}' without brackets -- "
              "was not propagated]")
    else:
        print("[no move token -- fine on a perception question, a Format "
              "Error on a move request]")
    _show_wrong_spans(record["target_text"], meta.get("wrong_spans", []))
    for span in meta.get("unverified_spans", []):
        print(f'!!! UNVERIFIED wrong span (not in the player reply): "{span}"')

    # --- analyst exchange (separate file; may be absent for this move)
    a_key = (meta["session_id"], meta["game_index"], meta["move_index"])
    a_rec = _state["analyst"].get(a_key)
    if a_rec is None:
        print("\n[no analyst record for this move -- datagen skips rounds "
              "whose analysis ended in a truncated search call]")
    else:
        print(f"\n=== Analyst question: {a_rec['meta']['question']}")
        print(f"Analyst: {a_rec['target_text']}")
    rating = meta.get("rating")
    print(f"\n[RATING: {rating if rating is not None else 'MISSING'}]")


def _show_game_end():
    meta = _state["moves"][-1]["meta"]
    won = meta.get("game_won")
    print(f"\n*** end of game: {'WON' if won else 'lost'} in "
          f"{len(_state['moves'])} recorded move(s). ***")
    print(">>> Pick another game (or run) above and press Reset.")


def on_reset(_):
    """Load the selected game and start stepping from move 0."""
    _refresh_games()  # re-read files: a run may still be growing
    key = game_dd.value
    with out:
        clear_output()
        if key is None or key not in _state.get("all_games", {}):
            print("No game selected (is the run empty?).")
            next_btn.disabled = True
            return
        _state["moves"] = _state["all_games"][key]
        _state["pos"] = 0
        next_btn.disabled = False
        print(f"Run '{run_dd.value}', {_game_option(key, _state['moves'])}.")
        print(">>> Press Next move to step through it.")


def on_next(_):
    next_btn.disabled = True
    try:
        with out:
            if _state["pos"] >= len(_state["moves"]):
                _show_game_end()
                return  # stays disabled until Reset
            _show_move(_state["moves"][_state["pos"]])
            _state["pos"] += 1
        next_btn.disabled = False
    except Exception:
        next_btn.disabled = False
        raise


run_dd.observe(_refresh_games, names="value")
reset_btn.on_click(on_reset)
next_btn.on_click(on_next)

# Controls on top, conversation in the middle, the one button at the bottom
# -- after a long game the button sits where you finished reading.
display(widgets.VBox([
    widgets.HBox([run_dd, reset_btn]),
    game_dd,
    out,
    next_btn,
]))

_refresh_games()
on_reset(None)